# RAG Assignment: Baseline (Bi-Encoder) vs. Re-Ranking (Cross-Encoder)



This notebook builds two retrieval pipelines over the **same chunked corpus** from a single PDF (`data.pdf`):

| | Pipeline 1 – Baseline | Pipeline 2 – Re-Ranking |
|---|---|---|
| Stage 1 | Bi-Encoder → FAISS → **Top 10** | *same* Top 10 from Pipeline 1 |
| Stage 2 | take Top 3 directly | Cross-Encoder scores all 10 (query, chunk) pairs → re-sort → **Top 3** |
| Generation | LLM → **Answer 1** | LLM (identical prompt) → **Answer 2** |


## 1. Install dependencies

In [17]:
# sentence-transformers : Bi-Encoder + Cross-Encoder models
# faiss-cpu             : vector store (exact inner-product index)
# pymupdf               : PDF text extraction
# google-genai          : Gemini API (LLM). A local Hugging Face model is used as a fallback.
!pip install -q sentence-transformers faiss-cpu pymupdf google-genai pandas

## 2. Imports & configuration

In [18]:
import os, re, time, textwrap

import numpy as np
import pandas as pd
import torch
import fitz                     # PyMuPDF
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder

pd.set_option("display.max_colwidth", 120)

# ---------------- Configuration ----------------
PDF_PATH = "data.pdf"

# Bi-Encoder: encodes query and chunk *independently* into vectors
BI_ENCODER_NAME = "sentence-transformers/all-MiniLM-L6-v2"
# Cross-Encoder: reads (query, chunk) *jointly* and outputs one relevance score
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

CHUNK_SIZE = 800        # max characters per chunk (~180-200 tokens, fits the 256-token bi-encoder limit)
CHUNK_OVERLAP = 150     # characters of trailing sentences repeated in the next chunk

TOP_K_RETRIEVE = 10     # candidates from the bi-encoder
TOP_N_CONTEXT = 3       # chunks passed to the LLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cuda


## 3. Load the PDF

In [19]:
# If data.pdf is not in the Colab working directory, ask the user to upload it.
if not os.path.exists(PDF_PATH):
    try:
        from google.colab import files
        print(f"'{PDF_PATH}' not found - please upload your PDF.")
        uploaded = files.upload()
        uploaded_name = next(iter(uploaded))
        if uploaded_name != PDF_PATH:
            os.rename(uploaded_name, PDF_PATH)
    except ImportError:
        raise FileNotFoundError(f"Place your PDF at ./{PDF_PATH}")
print("Using:", PDF_PATH)

Using: data.pdf


In [20]:
def clean_text(text: str) -> str:
    # Normalise raw PDF text: re-join hyphenated line breaks and collapse whitespace.
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)   # "regu-\nlation" -> "regulation"
    text = re.sub(r"\s+", " ", text)                # newlines / tabs / runs of spaces -> one space
    return text.strip()

def load_pdf(path: str):
    # Return a list of {"page": n, "text": ...} (1-indexed pages), skipping empty pages.
    doc = fitz.open(path)
    pages = []
    for i, page in enumerate(doc):
        text = clean_text(page.get_text("text"))
        if text:
            pages.append({"page": i + 1, "text": text})
    doc.close()
    return pages

pages = load_pdf(PDF_PATH)
print(f"Pages with text: {len(pages)}")
print(f"Total characters: {sum(len(p['text']) for p in pages):,}")
print("\nFirst 500 chars of page 1:\n", pages[0]["text"][:500])

Pages with text: 44
Total characters: 196,469

First 500 chars of page 1:
 ® Version 2.0


## 4. Chunking

Sentence-aware chunking: sentences are packed greedily into chunks of up to `CHUNK_SIZE` characters, and the last
~`CHUNK_OVERLAP` characters of sentences are repeated at the start of the next chunk so that a rule split across a
boundary is not lost. Chunks may cross page boundaries; each chunk records its page range for citation.

Both pipelines use **exactly this list of chunks**.

In [21]:
def split_sentences(text: str):
    # Lightweight sentence splitter (period / ! / ? followed by whitespace).
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]

def chunk_document(pages, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    # 1) Flatten the document into (sentence, page) units.
    units = []
    for p in pages:
        for sent in split_sentences(p["text"]):
            # Hard-split pathological "sentences" (tables, lists without periods) that exceed chunk_size.
            while len(sent) > chunk_size:
                cut = sent.rfind(" ", 0, chunk_size)
                cut = cut if cut > 0 else chunk_size
                units.append((sent[:cut].strip(), p["page"]))
                sent = sent[cut:].strip()
            if sent:
                units.append((sent, p["page"]))

    # 2) Greedily pack units into chunks, carrying an overlap tail into the next chunk.
    raw_chunks, current, cur_len = [], [], 0
    for sent, pg in units:
        if current and cur_len + len(sent) + 1 > chunk_size:
            raw_chunks.append(current)
            tail, tail_len = [], 0
            for u in reversed(current):
                if tail_len + len(u[0]) + 1 > overlap:
                    break
                tail.insert(0, u)
                tail_len += len(u[0]) + 1
            if tail_len + len(sent) + 1 > chunk_size:   # overlap + new sentence would not fit
                tail, tail_len = [], 0
            current, cur_len = tail, tail_len
        current.append((sent, pg))
        cur_len += len(sent) + 1
    if current:
        raw_chunks.append(current)

    # 3) Build chunk records.
    return [
        {
            "chunk_id": i,
            "text": " ".join(s for s, _ in c),
            "page_start": c[0][1],
            "page_end": c[-1][1],
        }
        for i, c in enumerate(raw_chunks)
    ]

chunks = chunk_document(pages)
chunk_texts = [c["text"] for c in chunks]

lengths = pd.Series([len(t) for t in chunk_texts])
print(f"Number of chunks: {len(chunks)}")
print("Chunk length (chars):", lengths.describe()[["min", "mean", "max"]].round(0).to_dict())

def page_label(c):
    return f"p.{c['page_start']}" if c["page_start"] == c["page_end"] else f"p.{c['page_start']}-{c['page_end']}"

for c in chunks[:3]:
    print(f"\n--- Chunk {c['chunk_id']} ({page_label(c)}) ---\n{textwrap.fill(c['text'][:400], 110)} ...")

Number of chunks: 304
Chunk length (chars): {'min': 106.0, 'mean': 720.0, 'max': 799.0}

--- Chunk 0 (p.1-2) ---
® Version 2.0 Content that is new to the current version of of Living Rules Reference is denoted by red text.
LIVING RULES REFERENCE VERSION 2.0 09/22/20 NEW GLOSSARY SECTIONS ✦Capture . . . . . . . . . . . . . . . . . .
. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 17 ✦Deploy . . . . . . . . . . . . . . . . . .
. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 30 ✦ ...

--- Chunk 1 (p.2) ---
. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 44 ✦Leader Sheet . . . . . . . . . . . . . . .
. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 50 ✦Leaders . . . . . . . . . . . . . . . . . . .
. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 51 ✦Legendary Planets . . . . . . . . . . . . .
. . . . . . . . . . . . . . . . . . . . . . . . . . 53 ✦Mechs . . . . ...

--- Chunk 2 (p.2) ---
. . . . . . . . . . . . . . . .

## 5. Pipeline 1 – Bi-Encoder indexing (vector store)

Each chunk is embedded **once, offline** into a 384-d vector. Embeddings are L2-normalised, so the inner product in
`faiss.IndexFlatIP` equals **cosine similarity**. At query time only the query needs to be embedded.

In [22]:
bi_encoder = SentenceTransformer(BI_ENCODER_NAME, device=DEVICE)
print("Bi-encoder max_seq_length (tokens):", bi_encoder.max_seq_length)

# Sanity check: how many chunks would be truncated by the bi-encoder's token limit?
n_tokens = [len(bi_encoder.tokenizer.encode(t, add_special_tokens=True)) for t in chunk_texts]
n_trunc = sum(n > bi_encoder.max_seq_length for n in n_tokens)
print(f"Chunks longer than the token limit (would be truncated): {n_trunc}/{len(chunks)}")

t0 = time.perf_counter()
chunk_embeddings = bi_encoder.encode(
    chunk_texts,
    batch_size=64,
    convert_to_numpy=True,
    normalize_embeddings=True,      # unit vectors -> inner product == cosine similarity
    show_progress_bar=True,
).astype("float32")
index_build_time = time.perf_counter() - t0

index = faiss.IndexFlatIP(chunk_embeddings.shape[1])   # exact (brute-force) inner-product search
index.add(chunk_embeddings)
print(f"Indexed {index.ntotal} chunks, dim={chunk_embeddings.shape[1]}, in {index_build_time:.2f}s (one-time cost)")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (324 > 256). Running this sequence through the model will result in indexing errors


Bi-encoder max_seq_length (tokens): 256
Chunks longer than the token limit (would be truncated): 27/304


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Indexed 304 chunks, dim=384, in 0.86s (one-time cost)


In [23]:
def baseline_retrieve(query: str, k: int = TOP_K_RETRIEVE):
    # Pipeline 1 (a,b): embed the query and return the top-k chunks by cosine similarity.
    q_emb = bi_encoder.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q_emb, k)
    return [
        {"chunk_id": int(cid), "bi_rank": r + 1, "bi_score": float(s)}
        for r, (cid, s) in enumerate(zip(ids[0], scores[0]))
    ]

## 6. Pipeline 2 – Cross-Encoder re-ranking

The cross-encoder concatenates `[CLS] query [SEP] chunk [SEP]` and runs full self-attention over both, producing one
relevance score per pair. It cannot pre-compute anything per chunk, so it is only applied to the 10 candidates.

In [24]:
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME, max_length=512, device=DEVICE)

def rerank(query: str, candidates):
    # Pipeline 2 (b,c): score every (query, chunk) pair and re-sort descending by cross-encoder score.
    pairs = [(query, chunks[c["chunk_id"]]["text"]) for c in candidates]
    ce_scores = cross_encoder.predict(pairs, batch_size=16, show_progress_bar=False)
    scored = [{**c, "ce_score": float(s)} for c, s in zip(candidates, ce_scores)]
    scored.sort(key=lambda x: x["ce_score"], reverse=True)
    for r, c in enumerate(scored):
        c["ce_rank"] = r + 1
    return scored

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

## 7. LLM setup

**Option A – Gemini (recommended):** create a free key at <https://aistudio.google.com/apikey>, then in Colab click
the 🔑 *Secrets* icon in the left sidebar → add `GOOGLE_API_KEY` → enable notebook access.

**Option B – local fallback:** if no key is found, a small open-weights model from Hugging Face is loaded on the GPU.

Temperature is 0 so that any difference between Answer 1 and Answer 2 comes from the **context**, not sampling.

In [25]:
GEMINI_MODEL = "gemini-3.5-flash-lite"              # change if a newer model is available to you
HF_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"        # local fallback (fits a T4 easily)

try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
except Exception:
    GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")

LLM_BACKEND = "gemini" if GOOGLE_API_KEY else "hf"
print("LLM backend:", LLM_BACKEND)

if LLM_BACKEND == "gemini":
    from google import genai
    from google.genai import types
    gemini_client = genai.Client(api_key=GOOGLE_API_KEY)

    def generate(prompt: str) -> str:
        resp = gemini_client.models.generate_content(
            model=GEMINI_MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(temperature=0.0),
        )
        return (resp.text or "").strip()
else:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    hf_tok = AutoTokenizer.from_pretrained(HF_MODEL)
    hf_model = AutoModelForCausalLM.from_pretrained(
        HF_MODEL,
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
        device_map="auto",
    )

    def generate(prompt: str) -> str:
        msgs = [{"role": "user", "content": prompt}]
        inputs = hf_tok.apply_chat_template(
            msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True
        ).to(hf_model.device)
        with torch.no_grad():
            out = hf_model.generate(**inputs, max_new_tokens=400, do_sample=False)
        return hf_tok.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

print(generate("Reply with the single word: ready"))

LLM backend: gemini
Ready


### Prompt template (identical for both pipelines)
The only thing that differs between Answer 1 and Answer 2 is **which 3 chunks** fill the context slot.

**Version 2 change:** the refusal line (*"If the context does not contain the answer, reply exactly..."*) and the word "ONLY" are removed. Everything else is the same as Version 1.

In [26]:
# Version 2: no refusal clause - the model must answer based on the given context
PROMPT_TEMPLATE = '''You are a careful assistant answering questions about a single document.
Answer the question based on the context passages below.
- Mention any exceptions, conditions, or restrictions stated in the context.
- Cite the passages you used, e.g. [Chunk 12].

Context:
{context}

Question: {question}

Answer:'''

def build_prompt(query: str, selected):
    context = "\n\n".join(
        f"[Chunk {c['chunk_id']} | {page_label(chunks[c['chunk_id']])}]\n{chunks[c['chunk_id']]['text']}"
        for c in selected
    )
    return PROMPT_TEMPLATE.format(context=context, question=query)

## 8. Run both pipelines on a query

`compare_pipelines(query)`:
1. Bi-encoder retrieves the Top 10 (timed).
2. **Pipeline 1:** Top 3 of those → LLM → Answer 1.
3. **Pipeline 2:** same Top 10 → cross-encoder → re-sorted Top 3 → LLM → Answer 2.

Every stage is timed separately so retrieval cost is not mixed up with LLM/network latency.

In [27]:
def compare_pipelines(query: str, generate_answers: bool = True):
    # --- Stage 1: bi-encoder retrieval (shared by both pipelines) ---
    t0 = time.perf_counter()
    candidates = baseline_retrieve(query, TOP_K_RETRIEVE)
    t_retrieve = time.perf_counter() - t0

    # --- Pipeline 1: take the top 3 directly ---
    p1_top = candidates[:TOP_N_CONTEXT]

    # --- Pipeline 2: cross-encoder re-rank of the same 10 candidates ---
    t0 = time.perf_counter()
    reranked = rerank(query, candidates)
    t_rerank = time.perf_counter() - t0
    p2_top = reranked[:TOP_N_CONTEXT]

    result = {
        "query": query, "candidates": candidates, "reranked": reranked,
        "p1_top": p1_top, "p2_top": p2_top,
        "t_retrieve": t_retrieve, "t_rerank": t_rerank,
    }

    if generate_answers:
        t0 = time.perf_counter()
        result["answer1"] = generate(build_prompt(query, p1_top))
        result["t_llm1"] = time.perf_counter() - t0

        t0 = time.perf_counter()
        result["answer2"] = generate(build_prompt(query, p2_top))
        result["t_llm2"] = time.perf_counter() - t0
    return result


def rank_shift_table(result):
    # One row per candidate: bi-encoder rank/score vs cross-encoder rank/score.
    rows = []
    for c in result["reranked"]:
        ch = chunks[c["chunk_id"]]
        rows.append({
            "chunk_id": c["chunk_id"],
            "page": page_label(ch),
            "bi_rank": c["bi_rank"],
            "bi_score": round(c["bi_score"], 4),
            "ce_rank": c["ce_rank"],
            "ce_score": round(c["ce_score"], 4),
            "shift": c["bi_rank"] - c["ce_rank"],          # positive = promoted by the cross-encoder
            "top3_P1": c["bi_rank"] <= TOP_N_CONTEXT,
            "top3_P2": c["ce_rank"] <= TOP_N_CONTEXT,
            "preview": ch["text"][:110] + "...",
        })
    return pd.DataFrame(rows).sort_values("bi_rank").reset_index(drop=True)


def show_result(result, show_chunks: bool = True):
    print("=" * 110)
    print("QUERY:", result["query"])
    print("=" * 110)

    df = rank_shift_table(result)
    display(df)

    # Human-readable rank movements into / out of the Top 3
    print("\nRank changes affecting the Top 3:")
    for _, r in df.iterrows():
        # use r["shift"], not r.shift: a pandas row already has a built-in .shift() method
        if r["top3_P1"] != r["top3_P2"] or (r["top3_P2"] and r["shift"] != 0):
            arrow = "promoted" if r["shift"] > 0 else "demoted"
            print(f"  Chunk {r.chunk_id} ({r.page}) moved from Rank #{r.bi_rank} -> Rank #{r.ce_rank}  [{arrow}]")
    overlap = {c["chunk_id"] for c in result["p1_top"]} & {c["chunk_id"] for c in result["p2_top"]}
    print(f"  Top-3 overlap between pipelines: {len(overlap)}/{TOP_N_CONTEXT}")

    if show_chunks:
        for name, key in [("PIPELINE 1 (Bi-Encoder) Top 3", "p1_top"), ("PIPELINE 2 (Cross-Encoder) Top 3", "p2_top")]:
            print(f"\n---------- {name} ----------")
            for c in result[key]:
                ch = chunks[c["chunk_id"]]
                print(f"\n[Chunk {c['chunk_id']} | {page_label(ch)} | bi_rank={c['bi_rank']} | ce_rank={c.get('ce_rank', '-')}]")
                print(textwrap.fill(ch["text"], 110))

    if "answer1" in result:
        print("\n" + "#" * 110)
        print("ANSWER 1 (Baseline):\n" + textwrap.fill(result["answer1"], 110, replace_whitespace=False))
        print("\nANSWER 2 (Re-ranked):\n" + textwrap.fill(result["answer2"], 110, replace_whitespace=False))
        print("#" * 110)

    # Per-query timing
    t = result
    timing = pd.DataFrame({
        "Stage": ["Bi-encoder retrieval (Top 10)", "Cross-encoder re-rank (10 pairs)", "LLM generation"],
        "Pipeline 1 (ms)": [t["t_retrieve"] * 1e3, 0.0, t.get("t_llm1", np.nan) * 1e3],
        "Pipeline 2 (ms)": [t["t_retrieve"] * 1e3, t["t_rerank"] * 1e3, t.get("t_llm2", np.nan) * 1e3],
    })
    timing.loc[len(timing)] = ["TOTAL", timing["Pipeline 1 (ms)"].sum(), timing["Pipeline 2 (ms)"].sum()]
    display(timing.round(2))

### Example query


In [28]:
QUERY = "Can a player produce ships in a system that contains another player's ships?"
result = compare_pipelines(QUERY)
show_result(result)

QUERY: Can a player produce ships in a system that contains another player's ships?


,chunk_id,page,bi_rank,bi_score,ce_rank,ce_score,shift,top3_P1,top3_P2,preview
0,163,p.26,1,0.5598,1,7.9677,0,True,True,"A player can remove any number of their units in this way; however, any units that are removed must be produce..."
1,145,p.23,2,0.5273,8,2.8292,-6,True,False,c The ship cannot move if it started its movement in another system that contains one of its faction’s command...
2,175,p.27-28,3,0.5238,9,2.2328,-6,True,False,"a Other players’ promissory notes are returned to those players. RELATED TOPICS: Abilities, Elimination, Neigh..."
3,185,p.29,4,0.5213,4,4.4919,0,False,False,77.4 This ability can be used even if no ships were moved during the “Move Ships” step. 77.5 STEP 2­—ASSIGN HI...
4,172,p.27,5,0.4989,2,6.4790,3,False,True,"68.2 When a player produces ships by using “Production,” that player must place them in the active system. 68...."
5,213,p.33,6,0.4905,6,3.7089,0,False,False,That system is the active system. b Other players’ command tokens do not prevent a player from activating a sy...
6,235,p.36,7,0.4900,10,-1.3023,-3,False,False,"a Players do not need to be neighbors to perform these transactions. RELATED TOPICS: Commodities, Deals, Neigh..."
7,144,p.23,8,0.4797,5,4.2843,3,False,False,"Additionally, some abilities can move a unit outside of the tactical action. 58.2 TACTICAL ACTION MOVEMENT 58...."
8,195,p.30-31,9,0.4663,3,5.6023,6,False,True,"Additionally, the system cannot contain ships controlled by another player. d If any of a player’s units succe..."
9,106,p.18,10,0.4505,7,3.0781,3,False,False,37.1 The number of command tokens in a player’s fleet pool indicates the maximum number of non-fighter ships t...



Rank changes affecting the Top 3:
  Chunk 145 (p.23) moved from Rank #2 -> Rank #8  [demoted]
  Chunk 175 (p.27-28) moved from Rank #3 -> Rank #9  [demoted]
  Chunk 172 (p.27) moved from Rank #5 -> Rank #2  [promoted]
  Chunk 195 (p.30-31) moved from Rank #9 -> Rank #3  [promoted]
  Top-3 overlap between pipelines: 1/3

---------- PIPELINE 1 (Bi-Encoder) Top 3 ----------

[Chunk 163 | p.26 | bi_rank=1 | ce_rank=-]
A player can remove any number of their units in this way; however, any units that are removed must be
produced immediately. b When producing a fighter or infantry unit, a player can use a fighter or infantry
token, as appropriate, from the supply instead of a plastic piece. 67.6 A player cannot produce ships in a
system that contains other players’ ships. a Ground forces can still be produced. RELATED TOPICS: Blockaded,
Cost, Fighter Tokens, Infantry Tokens, Production, Space Dock, Tactical Action 68 PRODUCTION (UNIT ABILITY)
During the “Production” step of a tactical actio

,Stage,Pipeline 1 (ms),Pipeline 2 (ms)
0,Bi-encoder retrieval (Top 10),15.06,15.06
1,Cross-encoder re-rank (10 pairs),0.00,41.09
2,LLM generation,903.34,964.25
3,TOTAL,918.39,1020.40


## 9. Latency profiling (retrieval vs. re-ranking)

LLM time depends on the network/API, so the retrieval stages are profiled separately. Each stage is repeated
`N_RUNS` times after a warm-up call (the first GPU call includes CUDA initialisation and would skew the numbers).

In [29]:
def profile_latency(queries, n_runs: int = 20):
    # Warm-up
    for q in queries:
        rerank(q, baseline_retrieve(q))

    rows = []
    for q in queries:
        t_ret, t_rr = [], []
        for _ in range(n_runs):
            t0 = time.perf_counter(); cands = baseline_retrieve(q); t_ret.append(time.perf_counter() - t0)
            t0 = time.perf_counter(); rerank(q, cands);            t_rr.append(time.perf_counter() - t0)
        t_ret, t_rr = np.array(t_ret) * 1e3, np.array(t_rr) * 1e3
        rows.append({
            "query": q[:60] + ("..." if len(q) > 60 else ""),
            "P1 retrieval mean (ms)": t_ret.mean(),
            "P1 retrieval std (ms)": t_ret.std(),
            "P2 re-rank only mean (ms)": t_rr.mean(),
            "P2 total retrieval mean (ms)": (t_ret + t_rr).mean(),
            "slowdown (x)": (t_ret + t_rr).mean() / t_ret.mean(),
        })
    return pd.DataFrame(rows).round(2)

N_RUNS = 20
latency_df = profile_latency([QUERY], n_runs=N_RUNS)
print(f"Device: {DEVICE} | chunks in index: {len(chunks)} | candidates re-ranked: {TOP_K_RETRIEVE}")
display(latency_df)

Device: cuda | chunks in index: 304 | candidates re-ranked: 10


,query,P1 retrieval mean (ms),P1 retrieval std (ms),P2 re-rank only mean (ms),P2 total retrieval mean (ms),slowdown (x)
0,Can a player produce ships in a system that contains another...,21.21,8.17,56.9,78.11,3.68


## 10. Helper for the next step (failure analysis)

`find_chunks("keyword")` lists chunks containing a phrase - useful for identifying the *ground-truth* chunk for a
query, so you can check where it sat in the bi-encoder ranking vs. after re-ranking.

In [30]:
def find_chunks(phrase: str, max_results: int = 10):
    hits = [c for c in chunks if phrase.lower() in c["text"].lower()]
    for c in hits[:max_results]:
        i = c["text"].lower().index(phrase.lower())
        snippet = c["text"][max(0, i - 120): i + 180]
        print(f"[Chunk {c['chunk_id']} | {page_label(c)}] ...{snippet}...\n")
    print(f"{len(hits)} chunk(s) contain '{phrase}'.")

# Example:
# find_chunks("penalty")

### 11.1 Two specific failure queries

| # | Query | Answer rule | Why the bi-encoder struggles |
|---|---|---|---|
| 1 | Can I activate a system where I already have a command token? | 5.2 (p.9) | A near-opposite rule (89.1b *"Other players' command tokens do not prevent..."*) shares almost all the query words |
| 2 | Can I spend resources from a planet I already tapped this round? | 64.9 (p.25) | The query says "tapped"; the rulebook only says "exhausted" |

**Gold phrase:** a short piece of text copied word-for-word from the rule that answers the query. It is used **only
for evaluation**: `gold_chunk_ids()` finds the chunk(s) containing it (the ground-truth chunks), and the summary table
reports where those chunks ranked before and after re-ranking. It is never given to the retrievers or the LLM.
Because neighbouring chunks overlap, one phrase can appear in two chunks (e.g. 155 and 156).

In [31]:
FAILURE_QUERIES = [
    {"query": "Can I activate a system where I already have a command token?",
     "gold_phrase": "cannot activate a system that already contains one of"},
    {"query": "Can I spend resources from a planet I already tapped this round?",
     "gold_phrase": "cannot spend an exhausted planet"},
]

def gold_chunk_ids(phrase):
    # Chunks whose text contains the answer phrase (curly apostrophes normalised).
    norm = lambda s: re.sub(r"\s+", " ", s.lower().replace("\u2019", "'"))
    return [c["chunk_id"] for c in chunks if norm(phrase) in norm(c["text"])]

failure_results, summary_rows = {}, []
for item in FAILURE_QUERIES:
    q = item["query"]
    res = compare_pipelines(q)
    failure_results[q] = res
    show_result(res)

    # Rank of each ground-truth chunk before (bi-encoder) and after (cross-encoder) re-ranking
    bi_ranks = {c["chunk_id"]: c["bi_rank"] for c in res["candidates"]}
    ce_ranks = {c["chunk_id"]: c["ce_rank"] for c in res["reranked"]}
    for gid in gold_chunk_ids(item["gold_phrase"]):
        summary_rows.append({
            "query": q,
            "gold_chunk": gid,
            "page": page_label(chunks[gid]),
            "bi_rank": bi_ranks.get(gid, "not in top 10"),
            "ce_rank": ce_ranks.get(gid, "not in top 10"),
            "in P1 top 3": bi_ranks.get(gid, 99) <= TOP_N_CONTEXT,
            "in P2 top 3": ce_ranks.get(gid, 99) <= TOP_N_CONTEXT,
        })

print("\nGROUND-TRUTH CHUNK RANK SHIFT")
display(pd.DataFrame(summary_rows))

QUERY: Can I activate a system where I already have a command token?


,chunk_id,page,bi_rank,bi_score,ce_rank,ce_score,shift,top3_P1,top3_P2,preview
0,244,p.37,1,0.5022,6,-2.4643,-5,True,False,"Then, that player gains that command token by placing it in a pool of their choice on their command sheet...."
1,213,p.33,2,0.4685,5,0.1085,-3,True,False,That system is the active system. b Other players’ command tokens do not prevent a player from activating a sy...
2,69,p.13,3,0.4554,3,1.7805,0,True,True,"a If a player would gain a command token but has none available in their reinforcements, that player cannot ga..."
3,94,p.16,4,0.4209,2,3.0060,2,False,True,"b If a player already has a command token in the chosen system, they do not place a command token there. 32.3 ..."
4,236,p.36-37,5,0.4162,9,-5.4262,-4,False,False,b Units being transported by a ship that is removed from the board by a gravity rift are also removed from the...
5,107,p.18,6,0.3993,7,-3.7828,-1,False,False,37.4 Players do not spend command tokens from this pool unless a game effect specifically allows it. RELATED T...
6,226,p.35,7,0.3678,8,-4.7688,-1,False,False,"Then, they place those tokens in the commodity area of their faction sheet. a A player cannot have more commod..."
7,48,p.9,8,0.3617,1,5.0122,7,False,True,"RELATED TOPICS: Action Phase, Initiative Order 5 ACTIVE SYSTEM The active system is the system that is activat..."
8,137,p.22,9,0.3575,10,-8.8969,-1,False,False,"54.1 During setup, the custodians token is placed on Mecatol Rex. This token prevents a player from committing..."
9,212,p.33,10,0.3569,4,0.2398,6,False,False,88.7 Double-sided tiles that have lines crossing from one edge to another are hyperlane tiles. Hyperlane tiles...



Rank changes affecting the Top 3:
  Chunk 244 (p.37) moved from Rank #1 -> Rank #6  [demoted]
  Chunk 213 (p.33) moved from Rank #2 -> Rank #5  [demoted]
  Chunk 94 (p.16) moved from Rank #4 -> Rank #2  [promoted]
  Chunk 48 (p.9) moved from Rank #8 -> Rank #1  [promoted]
  Top-3 overlap between pipelines: 1/3

---------- PIPELINE 1 (Bi-Encoder) Top 3 ----------

[Chunk 244 | p.37 | bi_rank=1 | ce_rank=-]
Then, that player gains that command token by placing it in a pool of their choice on their command sheet.

[Chunk 213 | p.33 | bi_rank=2 | ce_rank=-]
That system is the active system. b Other players’ command tokens do not prevent a player from activating a
system. 89.2 STEP 2—MOVEMENT: The active player may move any number of ships that have a sufficient move value
from any number of systems that do not contain one of their command tokens into the active system, following
the rules for movement. a Ships that have capacity values can transport ground forces and fighters when
moving.

,Stage,Pipeline 1 (ms),Pipeline 2 (ms)
0,Bi-encoder retrieval (Top 10),15.10,15.10
1,Cross-encoder re-rank (10 pairs),0.00,48.84
2,LLM generation,1524.94,1094.20
3,TOTAL,1540.04,1158.14


QUERY: Can I spend resources from a planet I already tapped this round?


,chunk_id,page,bi_rank,bi_score,ce_rank,ce_score,shift,top3_P1,top3_P2,preview
0,181,p.28-29,1,0.3936,1,-1.2787,0,True,True,75.1 A planet’s resources are the leftmost value that is surrounded by a yellow border on the planet’s system ...
1,180,p.28,2,0.3815,5,-7.2892,-3,True,False,"73.4 Relics cannot be traded. RELATED TOPICS: Exploration, Purge 74 REROLLS Some game effects instruct a playe..."
2,85,p.15,3,0.3678,4,-6.7003,-1,True,False,"25.7 If a player loses control of a planet that contains their control token, they remove their control token ..."
3,156,p.25,4,0.3624,2,-1.7465,2,False,True,64.8 A player can spend a readied planet’s resources or influence. 64.9 A player cannot spend an exhausted pla...
4,155,p.25,5,0.3504,3,-4.8345,2,False,True,"64.5 Some planets are legendary planets, as indicated by the legendary planet icon. When a player gains contro..."
5,107,p.18,6,0.3377,6,-8.6981,0,False,False,37.4 Players do not spend command tokens from this pool unless a game effect specifically allows it. RELATED T...
6,160,p.25,7,0.3359,9,-9.9961,-2,False,False,63.3 A PDS unit is placed on a planet. Each planet can have a maximum of two PDS units. 63.4 If a player’s PDS...
7,108,p.18,8,0.3306,7,-9.5465,1,False,False,"c If a planet has multiple traits, the player exploring the planet chooses which of the corresponding explorat..."
8,90,p.15,9,0.2793,8,-9.8201,1,False,False,"a If a player is the first player to control a planet, they take the planet card from the planet card deck. b ..."
9,154,p.24-25,10,0.2758,10,-11.2967,0,False,False,a A player can score an objective during both the space combat and the ground combat during the same tactical ...



Rank changes affecting the Top 3:
  Chunk 180 (p.28) moved from Rank #2 -> Rank #5  [demoted]
  Chunk 85 (p.15) moved from Rank #3 -> Rank #4  [demoted]
  Chunk 156 (p.25) moved from Rank #4 -> Rank #2  [promoted]
  Chunk 155 (p.25) moved from Rank #5 -> Rank #3  [promoted]
  Top-3 overlap between pipelines: 1/3

---------- PIPELINE 1 (Bi-Encoder) Top 3 ----------

[Chunk 181 | p.28-29 | bi_rank=1 | ce_rank=-]
75.1 A planet’s resources are the leftmost value that is surrounded by a yellow border on the planet’s system
tile and planet card. 75.2 A player spends a planet’s resources by exhausting its card. 75.3 A player can
spend a trade good as if it were one resource. RELATED TOPICS: Exhausted, Planets, Producing Units, Trade
Goods 29 78 SPACE COMBAT After resolving the “Space Cannon Offense” step of a tactical action, if two players
have ships in the active system, those players must resolve a space combat. 78.1 If the active player is the
only player with ships in the system, they s

,Stage,Pipeline 1 (ms),Pipeline 2 (ms)
0,Bi-encoder retrieval (Top 10),13.67,13.67
1,Cross-encoder re-rank (10 pairs),0.00,42.42
2,LLM generation,1040.84,1998.00
3,TOTAL,1054.51,2054.10



GROUND-TRUTH CHUNK RANK SHIFT


,query,gold_chunk,page,bi_rank,ce_rank,in P1 top 3,in P2 top 3
0,Can I activate a system where I already have a command token?,48,p.9,8,1,False,True
1,Can I spend resources from a planet I already tapped this round?,155,p.25,5,3,False,True
2,Can I spend resources from a planet I already tapped this round?,156,p.25,4,2,False,True


### 11.2 Document-specific failure categories (1 query each)

| Category | Query | Answer rule | Why the bi-encoder struggles |
|---|---|---|---|
| Keyword trap / distractor | Do players who are not in the battle get to use combat abilities? | 62.1 (p.25) | The rule uses different wording ("no units on either side of a combat... not opponents"); generic "combat" chunks score higher |
| Exception / corner case | If my space dock is blockaded, can I still produce anything there? | 14.1 (p.11) | General blockade/capture chunks rank higher than the exception "can still produce ground forces" |
| Negation / constraint | Which hits can Sustain Damage not cancel? | 87.4a, 87.5 (p.33) | FAQ chunks about Sustain Damage outrank the rule that states what it *cannot* do |
| Buried at rank 4-10 | Can I still make trades on the turn that I pass? | 3.3a (p.8) | Trade-goods chunks rank higher; the passing rule sits at rank 6 |

Uses `gold_chunk_ids()` from 11.1, so run that cell first.

In [32]:
CATEGORY_QUERIES = [
    {"category": "Keyword trap", "query": "Do players who are not in the battle get to use combat abilities?",
     "gold_phrase": "Players who do not have units on either side of a combat are"},
    {"category": "Exception", "query": "If my space dock is blockaded, can I still produce anything there?",
     "gold_phrase": "can still use a blockaded unit to produce ground forces"},
    {"category": "Negation", "query": "Which hits can Sustain Damage not cancel?",
     "gold_phrase": "cannot be used to cancel an"},
    {"category": "Buried (rank 4-10)", "query": "Can I still make trades on the turn that I pass?",
     "gold_phrase": "During a turn that a player passes, they can resolve"},
]

category_results, category_rows = {}, []
for item in CATEGORY_QUERIES:
    q = item["query"]
    print("\n\n>>> CATEGORY:", item["category"])
    res = compare_pipelines(q)
    category_results[q] = res
    show_result(res)

    bi_ranks = {c["chunk_id"]: c["bi_rank"] for c in res["candidates"]}
    ce_ranks = {c["chunk_id"]: c["ce_rank"] for c in res["reranked"]}
    for gid in gold_chunk_ids(item["gold_phrase"]):
        category_rows.append({
            "category": item["category"],
            "query": q,
            "gold_chunk": gid,
            "page": page_label(chunks[gid]),
            "bi_rank": bi_ranks.get(gid, "not in top 10"),
            "ce_rank": ce_ranks.get(gid, "not in top 10"),
            "in P1 top 3": bi_ranks.get(gid, 99) <= TOP_N_CONTEXT,
            "in P2 top 3": ce_ranks.get(gid, 99) <= TOP_N_CONTEXT,
        })

print("\nGROUND-TRUTH CHUNK RANK SHIFT BY CATEGORY")
display(pd.DataFrame(category_rows))



>>> CATEGORY: Keyword trap
QUERY: Do players who are not in the battle get to use combat abilities?


,chunk_id,page,bi_rank,bi_score,ce_rank,ce_score,shift,top3_P1,top3_P2,preview
0,72,p.13,1,0.3824,8,-3.2529,-7,True,False,18 COMBAT (ATTRIBUTE) Combat is an attribute of some units that is presented on faction sheets and unit upgrad...
1,69,p.13,2,0.3824,10,-5.1404,-8,True,False,"a If a player would gain a command token but has none available in their reinforcements, that player cannot ga..."
2,215,p.33,3,0.3775,3,3.9945,0,True,True,a The active player may do this even if they did not move units or land ground forces during this tactical act...
3,96,p.16,4,0.3766,2,4.0239,2,False,True,"They cannot be part of a binding deal. 28.4 If the terms of a deal cannot be resolved immediately, it is a non..."
4,159,p.25,5,0.3726,1,6.9015,4,False,True,62.1 Players who do not have units on either side of a combat are not opponents. Those players cannot use abil...
5,61,p.11-12,6,0.3378,5,2.2253,1,False,False,"c Game effects that reroll, modify, or otherwise affect combat rolls do not affect anti-fighter barrage rolls...."
6,41,p.8,7,0.3359,9,-4.5178,-2,False,False,"a During a three-player or four-player game, a player cannot pass until they have exhausted both of their stra..."
7,64,p.12,8,0.3347,7,-1.4180,1,False,False,17.4 Captured fighters and infantry do not belong to any player and are returned only when an ability instruct...
8,193,p.30,9,0.3278,4,2.6809,5,False,False,f The attacker makes all of their combat rolls during this step before the defender. This procedure is importa...
9,171,p.27,10,0.3228,6,-0.4933,4,False,False,"71.6 When a player performs a strategic action, they exhaust their chosen strategy card. a That card is later ..."



Rank changes affecting the Top 3:
  Chunk 72 (p.13) moved from Rank #1 -> Rank #8  [demoted]
  Chunk 69 (p.13) moved from Rank #2 -> Rank #10  [demoted]
  Chunk 96 (p.16) moved from Rank #4 -> Rank #2  [promoted]
  Chunk 159 (p.25) moved from Rank #5 -> Rank #1  [promoted]
  Top-3 overlap between pipelines: 1/3

---------- PIPELINE 1 (Bi-Encoder) Top 3 ----------

[Chunk 72 | p.13 | bi_rank=1 | ce_rank=-]
18 COMBAT (ATTRIBUTE) Combat is an attribute of some units that is presented on faction sheets and unit
upgrade technology cards. 18.1 During combat, if a unit’s combat roll produces a result equal to or greater
than its combat value, it produces a hit. 18.2 If a unit’s combat value contains two or more burst icons,
instead of rolling a single die, the player rolls one die for each burst icon when making that unit’s combat
rolls. RELATED TOPICS: Ground Combat, Invasion, Space Combat 19 COMMAND SHEET Each player has a command sheet
that contains a strategy pool, a tactic pool, a fleet

,Stage,Pipeline 1 (ms),Pipeline 2 (ms)
0,Bi-encoder retrieval (Top 10),11.14,11.14
1,Cross-encoder re-rank (10 pairs),0.00,44.04
2,LLM generation,955.79,770.51
3,TOTAL,966.93,825.69




>>> CATEGORY: Exception
QUERY: If my space dock is blockaded, can I still produce anything there?


...

,chunk_id,page,bi_rank,bi_score,ce_rank,ce_score,shift,top3_P1,top3_P2,preview
0,229,p.35,1,0.3398,4,-8.0766,-3,True,False,93.5 A player can exchange their trade goods with other players during a transaction. 93.6 When a player recei...
1,228,p.35,2,0.3108,7,-9.8215,-5,True,False,93.1 Trade goods and commodities are represented by opposite sides of the same token. 93.2 When a player gains...
2,277,p.41,3,0.3062,1,-0.6571,2,True,True,"A: When those infantry would be placed on a home planet, they are returned to that player’s reinforcements ins..."
3,227,p.35,4,0.3000,6,-9.3945,-2,False,False,"b The chosen players can only use the secondary ability once, and they cannot use it by spending command token..."
4,83,p.14,5,0.2982,3,-6.6134,2,False,True,21.8 Commodity tokens come in values of one and three. A player can swap between these tokens as necessary. RE...
5,40,p.8,6,0.2868,2,-1.2066,4,False,True,"3.2 If a player cannot perform an action, they must pass. 3.3 After a player has passed, they have no further ..."
6,234,p.36,7,0.2779,10,-10.9871,-3,False,False,a The Emirates of Hacan can also exchange action cards with other players as part of their transactions. 94.4 ...
7,225,p.35,8,0.2716,9,-10.5297,-1,False,False,"RELATED TOPICS: Initiative Order, Resources, Strategic Action, Strategy Card, Technology 92 TRADE (STRATEGY CA..."
8,174,p.27,9,0.2711,5,-8.2398,4,False,False,a An unrevealed promissory note is not subject to effects in its ability text that return the card if certain ...
9,82,p.14,10,0.2578,8,-9.8245,2,False,False,a That token is no longer a commodity token; it is a trade good token. b A player can trade commodity tokens b...



Rank changes affecting the Top 3:
  Chunk 229 (p.35) moved from Rank #1 -> Rank #4  [demoted]
  Chunk 228 (p.35) moved from Rank #2 -> Rank #7  [demoted]
  Chunk 277 (p.41) moved from Rank #3 -> Rank #1  [promoted]
  Chunk 83 (p.14) moved from Rank #5 -> Rank #3  [promoted]
  Chunk 40 (p.8) moved from Rank #6 -> Rank #2  [promoted]
  Top-3 overlap between pipelines: 1/3

---------- PIPELINE 1 (Bi-Encoder) Top 3 ----------

[Chunk 229 | p.35 | bi_rank=1 | ce_rank=-]
93.5 A player can exchange their trade goods with other players during a transaction. 93.6 When a player
receives a commodity token from another player, the player who received that token places it in their trade
good area with the trade good side of the token faceup. a That token is no longer a commodity token; it is a
trade good token. 93.7 Trade good tokens come in values of one and three. A player can swap between these
tokens as necessary. RELATED TOPICS: Commodities, Deals, Influence, Resources, Transactions 94 TRANSA

,Stage,Pipeline 1 (ms),Pipeline 2 (ms)
0,Bi-encoder retrieval (Top 10),36.28,36.28
1,Cross-encoder re-rank (10 pairs),0.00,49.94
2,LLM generation,1156.29,2280.98
3,TOTAL,1192.57,2367.20



GROUND-TRUTH CHUNK RANK SHIFT BY CATEGORY


,category,query,gold_chunk,page,bi_rank,ce_rank,in P1 top 3,in P2 top 3
0,Keyword trap,Do players who are not in the battle get to use combat abilities?,159,p.25,5,1,False,True
1,Exception,"If my space dock is blockaded, can I still produce anything there?",58,p.11,4,1,False,True
2,Negation,Which hits can Sustain Damage not cancel?,210,p.33,4,1,False,True
3,Buried (rank 4-10),Can I still make trades on the turn that I pass?,40,p.8,6,2,False,True


In [33]:
ALL_QUERIES = [QUERY] + [q["query"] for q in FAILURE_QUERIES] + [q["query"] for q in CATEGORY_QUERIES]

latency_all = profile_latency(ALL_QUERIES, n_runs=N_RUNS)
print(f"Device: {DEVICE} | chunks in index: {len(chunks)} | candidates re-ranked: {TOP_K_RETRIEVE} "
      f"| queries: {len(ALL_QUERIES)} | runs per query: {N_RUNS}")
display(latency_all)

# Summary table for the report: average over all queries
summary = pd.DataFrame({
    "Pipeline": ["Pipeline 1 - Bi-encoder retrieval (Top 10 -> Top 3)",
                 "Pipeline 2 - Cross-encoder re-rank only (10 pairs)",
                 "Pipeline 2 - Total (retrieval + re-rank)"],
    "Mean latency (ms)": [latency_all["P1 retrieval mean (ms)"].mean(),
                          latency_all["P2 re-rank only mean (ms)"].mean(),
                          latency_all["P2 total retrieval mean (ms)"].mean()],
})
summary["Relative to Pipeline 1 (x)"] = summary["Mean latency (ms)"] / summary["Mean latency (ms)"].iloc[0]
print("\nAVERAGE OVER ALL QUERIES (excluding LLM generation)")
display(summary.round(2))

Device: cuda | chunks in index: 304 | candidates re-ranked: 10 | queries: 7 | runs per query: 20


,query,P1 retrieval mean (ms),P1 retrieval std (ms),P2 re-rank only mean (ms),P2 total retrieval mean (ms),slowdown (x)
0,Can a player produce ships in a system that contains another...,14.91,6.62,43.55,58.46,3.92
1,Can I activate a system where I already have a command token...,12.15,5.60,36.40,48.56,4.00
2,Can I spend resources from a planet I already tapped this ro...,11.59,3.12,33.82,45.41,3.92
3,Do players who are not in the battle get to use combat abili...,11.23,2.18,31.62,42.85,3.82
4,"If my space dock is blockaded, can I still produce anything ...",10.68,1.41,32.20,42.88,4.01
5,Which hits can Sustain Damage not cancel?,12.29,4.51,33.41,45.70,3.72
6,Can I still make trades on the turn that I pass?,12.02,9.23,36.43,48.45,4.03



AVERAGE OVER ALL QUERIES (excluding LLM generation)


,Pipeline,Mean latency (ms),Relative to Pipeline 1 (x)
0,Pipeline 1 - Bi-encoder retrieval (Top 10 -> Top 3),12.12,1.00
1,Pipeline 2 - Cross-encoder re-rank only (10 pairs),35.35,2.92
2,Pipeline 2 - Total (retrieval + re-rank),47.47,3.92
